In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from pyspark.sql.functions import (
    col,
    from_json,
    to_timestamp,
    coalesce,
    window,
    sum as Fsum,
    when,
    to_json,
    struct,
    date_format,
    round as Fround,
    expr
)

In [2]:
# Configuración
BOOTSTRAP_SERVERS="192.168.80.34:9092"
TOPIC_INPUT="gittba_BNB"
TOPIC_OUTPUT="gittba_BNB_VWAP"
CHECKPOINT_LOCATION = "/datos/gittba26/gittba05/checkpoint"

In [3]:
# 1. Crear la sesión de Spark
conf = (SparkConf()
            .setMaster("yarn")
            .set("spark.executor.cores", 5)
            .set("spark.sql.shuffle.partitions", 200)
            .set("spark.default.parallelism", 200)
            .set("spark.executor.memory", "7g")
            .set("spark.dynamicAllocation.maxExecutors", 20)
            .set("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6")
        )

spark = SparkSession.builder \
    .config(conf=conf) \
    .appName("CryptoVWAPCalculator_BNB") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

# Trabajamos en UTC para que las ventanas salgan bien
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/sharenfs/spark-3.5.6-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/alumnos/tecno_datos_masivos_gittba_bdt/gittba_bdt09/.ivy2/cache
The jars for the packages stored in: /home/alumnos/tecno_datos_masivos_gittba_bdt/gittba_bdt09/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a9e92e4e-a070-468f-8cf3-a99e0a12a05c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.6 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.6 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#comm

In [4]:
# 2. Leer datos del topic de Kafka de entrada
df_kafka_input = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS) \
    .option("subscribe", TOPIC_INPUT) \
    .option("startingOffsets", "latest") \
    .load()

In [5]:
# ==========================================
# 4. Esquema del JSON de entrada
# Ejemplo esperado:
# {
#   "symbol": "BNBUSDT",
#   "@timestamp": "2026-03-09T11:21:00Z",
#   "close": 67396.99,
#   "volume": 19.14504,
#   "timestamp": 1773650460000   # opcional si HU6 lo incluye
# }
# ==========================================
schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("@timestamp", StringType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("timestamp", LongType(), True)   # opcional
])

In [6]:
# =========================
# PARSE JSON
# =========================

df_parsed = (
    df_kafka_input
    .selectExpr(
        "CAST(key AS STRING) AS kafka_key",
        "CAST(value AS STRING) AS kafka_value"
    )
    .select(
        col("kafka_key"),
        from_json(col("kafka_value"), schema).alias("data")
    )
    .select(
        coalesce(col("data.symbol"), col("kafka_key")).alias("symbol"),
        coalesce(
            to_timestamp(col("data.`@timestamp`"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
            to_timestamp(col("data.`@timestamp`"), "yyyy-MM-dd'T'HH:mm:ssX")
        ).alias("event_time"),
        col("data.close").cast("double").alias("close"),
        col("data.volume").cast("double").alias("volume")
    )
    .where(
        col("symbol").isNotNull() &
        col("event_time").isNotNull() &
        col("close").isNotNull() &
        col("volume").isNotNull() &
        (col("volume") >= 0)
    )
)

In [7]:
# ==========================================
# 6. Calcular VWAP por ventanas de 5 minutos
# VWAP = sum(precio * volumen) / sum(volumen)
# ==========================================
df_vwap = (
    df_parsed
    .withWatermark("event_time", "1 minute")
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("symbol")
    )
    .agg(
        Fsum(expr("close * volume")).alias("sum_price_volume"),
        Fsum(col("volume")).alias("sum_volume")
    )
    .withColumn(
        "vwap",
        when(
            col("sum_volume") > 0,
            col("sum_price_volume") / col("sum_volume")
        )
    )
    .where(col("vwap").isNotNull())
)

In [8]:
# ==========================================
# 7. Formatear salida para Kafka
# Clave = símbolo
# Valor = JSON
# ==========================================
df_output = (
    df_vwap
    .select(
        col("symbol").alias("key"),
        to_json(
            struct(
                date_format(
                    col("window.start"),
                    "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"
                ).alias("window_start"),

                date_format(
                    col("window.end"),
                    "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"
                ).alias("window_end"),

                col("symbol").alias("symbol"),

                Fround(col("vwap"), 2).alias("vwap")
            )
        ).alias("value")
    )
    .selectExpr(
        "CAST(key AS STRING)",
        "CAST(value AS STRING)"
    )
)

In [9]:
# ==========================================
# 9. Escribir resultados en el topic Kafka de salida
# ==========================================
query = (
    df_output.writeStream
        .format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
        .option("topic", TOPIC_OUTPUT)
        .option("checkpointLocation", CHECKPOINT_LOCATION)
        .outputMode("update")
        .start()
)

26/03/25 19:53:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [10]:
# 8. Se queda esperando
query.awaitTermination()

26/03/25 19:53:49 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/25 19:55:28 WARN TaskSetManager: Lost task 113.1 in stage 1.0 (TID 205) (worker02.bigdata.alumnos.upcont.es executor 16): TaskKilled (another attempt succeeded)
26/03/25 19:55:28 WARN TaskSetManager: Lost task 117.1 in stage 1.0 (TID 212) (worker02.bigdata.alumnos.upcont.es executor 15): TaskKilled (another attempt succeeded)
26/03/25 19:55:28 WARN TaskSetManager: Lost task 115.1 in stage 1.0 (TID 217) (worker02.bigdata.alumnos.upcont.es executor 16): TaskKilled (another attempt succeeded)
26/03/25 19:55:29 WARN TaskSetManager: Lost task 150.0 in stage 1.0 (TID 151) (worker02.bigdata.alumnos.upcont.es executor 9): TaskKilled (another attempt succeeded)
26/03/25 19:55:29 WARN TaskSetManager: Lost task 148.0 in stage 1.0 (TID 149) (worker02.bigdata.alumnos.upcont.es executor 9): TaskKille

KeyboardInterrupt: 

In [ ]:
query.stop()